In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
from google.colab import drive
import os

def parse_broken_row(line):
  line = line.strip()

  while line.endswith(';') or line.endswith(',') or line.endswith(' '):
    line = line[:-1]
  
  if line.startswith('"') and line.endswith('"'):
    line = line[1:-1]

  parts = line.split(',', 1)
  if len(parts) < 2:
    return None
  
  label = parts[0].strip()
  text = parts[1].strip()

  text = text.strip(' ,;')
  if text.startswith('""') and text.endswith('""'):
    text = text[2:-2].replace('""', '"')
  elif text.startswith('"') and text.endswith('"'):
    text = text[1:-1].replace('""', '"')
  
  return label, text


  return label, text

  return line

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
ENGLISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_kaggle.csv')
SPANISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')

BATCH_SIZE = 100
NAMES = ['label', 'text']
MODEL_NAME = 'Helsinki-NLP/opus-mt-en-es'

data = []
try:
  with open(ENGLISH_FILE, 'r', encoding='latin-1') as file:
    for line in file:
      parsed_row = parse_broken_row(line)
      if parsed_row:
        label, text = parsed_row
        if label in ['ham', 'spam']:
          data.append({'label': label, 'text': text})
    dataset = pd.DataFrame(data)
    print(f"Dataset loaded successfully. Total rows: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

print(f"Loading model {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

def translate_batch(texts, tokenizer, model, device):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

print("Starting translation...")

spanish_texts = []
total_texts = len(dataset)

for i in tqdm(range(0, total_texts, BATCH_SIZE)):
    batch_texts = dataset['text'].iloc[i:i + BATCH_SIZE].tolist()
    translated_texts = translate_batch(batch_texts, tokenizer, model, device)
    spanish_texts.extend(translated_texts)

dataset['text_es'] = spanish_texts
print("Example translations:")
print(dataset[['text', 'text_es']].head())

dataset_final = dataset[['label', 'text_es']].rename(columns={'text_es': 'text'})
dataset_final.to_csv(SPANISH_FILE, index=False)

print(f"Translation completed. Translated dataset saved to {SPANISH_FILE}.")

In [ ]:
import html

BASE_PATH = '/content/drive/My Drive/TFG_Posdata'
SPANISH_FILE_RAW = os.path.join(BASE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')
SPANISH_FILE_CLEAN = os.path.join(BASE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_clean.csv')

print(f"Loading {SPANISH_FILE_RAW}...")
df = pd.read_csv(SPANISH_FILE_RAW)

def clean_text(text):
    if not isinstance(text, str):
        return str(text)
    
    text = html.unescape(text)
    
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
        
    return text

print("Cleaning text...")
df['text'] = df['text'].apply(clean_text)

df.to_csv(SPANISH_FILE_CLEAN, index=False, encoding='utf-8-sig')

print(f"Done! Cleaned dataset saved as: {SPANISH_FILE_CLEAN}")
print("You can now download this new file and open it in Excel without issues.")